# Data Acquisition and Loading

## Objective
Load two complementary datasets for power grid anomaly detection:
1. **UCI Electrical Grid Stability** — 10,000 simulated 4-node grid observations
2. **Synthetic Power Plant Data** — 5,000 samples with 5 realistic fault modes

The dual-dataset approach enables evaluation across both grid-level stability prediction and plant-level fault diagnosis.

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))

from src.data_loader import (
    load_uci_grid_stability,
    generate_power_plant_data,
    create_combined_dataset,
    split_data,
    UCI_FEATURE_DESCRIPTIONS,
)

DATA_DIR = os.path.join('..', 'data')
print('Libraries loaded')

## 1.1 Load UCI Electrical Grid Stability Dataset

The UCI dataset models a 4-node star topology: 1 electricity producer connected to 3 consumers. Features include reaction times ($\tau$), nominal power ($p$), and price elasticity coefficients ($g$). The target indicates whether the grid configuration is dynamically **stable** or **unstable**.

In [ ]:
uci_df = load_uci_grid_stability(os.path.join(DATA_DIR, 'Data_for_UCI_named.csv'))
print(f'\nShape: {uci_df.shape}')
print(f'\nFeature descriptions:')
for feat, desc in UCI_FEATURE_DESCRIPTIONS.items():
    print(f'  {feat:12s} — {desc}')
uci_df.head()

## 1.2 Generate Synthetic Power Plant Data

Simulate operational data from a gas-turbine power plant with 13 monitored parameters. Anomalies are injected via 5 fault modes calibrated to real-world failure scenarios:

| Fault Mode | Mechanism | Affected Parameters |
|------------|-----------|--------------------|
| Overload | Excessive load beyond rated capacity | load, current, PF, vibration |
| Voltage Sag | Sudden voltage drop | voltage, current, active power |
| Frequency Deviation | Grid frequency excursion > ±0.5 Hz | frequency, RPM |
| Thermal Runaway | Uncontrolled temperature rise | exhaust temp, coolant temp, vibration |
| Sensor Drift | Gradual sensor offset accumulation | voltage, oil pressure |

In [ ]:
plant_df = generate_power_plant_data(n_samples=5000, anomaly_ratio=0.08, random_state=42)
print(f'\nShape: {plant_df.shape}')
print(f'\nColumns: {list(plant_df.columns)}')
plant_df.head()

In [ ]:
# Fault type distribution
print('Fault type distribution:')
print(plant_df['fault_type'].value_counts())

fig, ax = plt.subplots(figsize=(8, 5))
plant_df['fault_type'].value_counts().plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Synthetic Power Plant — Fault Type Distribution')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../results/figures/fault_type_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.3 Train/Test Splits

In [ ]:
# UCI dataset split
uci_features = ['tau1','tau2','tau3','tau4','p1','p2','p3','p4','g1','g2','g3','g4']
uci_df['anomaly'] = (uci_df['stabf'] == 'unstable').astype(int)

print('=== UCI Grid Stability ===')
X_train_uci, X_test_uci, y_train_uci, y_test_uci = split_data(
    uci_df, uci_features, target_col='anomaly'
)

In [ ]:
# Power plant split
plant_features = [c for c in plant_df.columns if c not in ['anomaly', 'fault_type', 'source']]

print('\n=== Synthetic Power Plant ===')
X_train_plant, X_test_plant, y_train_plant, y_test_plant = split_data(
    plant_df, plant_features, target_col='anomaly'
)

## 1.4 Save Processed Data

In [ ]:
# Save for downstream notebooks
uci_df.to_csv(os.path.join(DATA_DIR, 'uci_grid_processed.csv'), index=False)
plant_df.to_csv(os.path.join(DATA_DIR, 'power_plant_synthetic.csv'), index=False)

print('Saved:')
print(f'  data/uci_grid_processed.csv ({len(uci_df):,} rows)')
print(f'  data/power_plant_synthetic.csv ({len(plant_df):,} rows)')

## Summary

| Dataset | Samples | Features | Anomaly Rate | Anomaly Type |
|---------|---------|----------|--------------|-------------|
| UCI Grid Stability | 10,000 | 12 | ~36% (unstable) | Grid instability |
| Synthetic Power Plant | 5,000 | 16 | 8% | 5 fault modes |

